[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# A Complete Workflow


## What you will be able to do

Take a raw table from a messy export to a tested, saved model of January temperature, and report how
far off it is and where it does worst, with every number measured on stations it never saw.

## The idea

### The problem

A planning office wants to know how cold January will be at stations it has not built yet, and the
station register sends you its export of 1,200 existing stations. Some private stations have no
elevation on record, a few have no owner, and the terrain was typed by hand, so `inland` sometimes
arrives as `INLAND ` with a trailing space. In a week you will present a model to people who will
ask how far off it is, whether it is better than a simple guess, and where it should not be trusted.

Every piece needed is something this guide has already taught. The hard part is the order. A test
set looked at early stops being a test, a scaler fitted before the split has already seen the
stations it will be scored on, and the target left among the features makes a model look perfect.
None of those mistakes raises an error, and each one makes the final number less true.

### What a workflow is

> A **workflow** here is the sequence that turns a raw table into a model you can defend: clean what
> needs no learning, choose the target and the features, split once and set the test rows aside, put
> every step that learns inside a pipeline, choose between models by **cross-validation** on the
> training rows alone, score the chosen model on the test rows once, report what it gets wrong, and
> save it with the versions it needs.

### Why it works that way

Each step protects the one number the meeting will remember, the error on stations the model never
saw. Cleaning text that needs no learning, such as removing spaces and capitals, can happen before
the split, because it treats every row the same way whatever the other rows hold. Filling in a
missing elevation with a median, or scaling a column, learns from the rows it is given, so it goes
inside the pipeline, where cross-validation and the final fit give it the training rows only.

Choosing a model is a decision made with data, and the data it is made with cannot also judge it.
Cross-validation on the training rows does the choosing, and the test rows are opened once, after
the choice, so their score is an honest estimate for new stations.

A single error number is not a report. The same average can hide a model that is good by the fjords
and poor in the mountains, and a planning office that builds in the mountains needs to know.

### Where this shows up

Any model that someone else will rely on goes through these steps, whatever the library. The Visual
Handbook covers the statistics behind the evaluation, and every step here is one this guide taught,
from the **Train and Test** notebook's split to the **Saving a Model** notebook's file.

### What this notebook covers

- the raw export, and the cleaning that can happen before the split
- the target, the features, and the columns left out
- a pipeline that fills gaps, scales numbers and encodes text
- three models compared by cross-validation, and a small search
- the test set used once, a report of the errors, and the saved model
- four mistakes that raise nothing or raise late, each run

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things.

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = Pipeline([("prepare", prepare()), ("model", LinearRegression())])
scores = cross_validate(model, X_train, y_train, cv=folds, scoring="neg_mean_absolute_error")
print("cross-validated error:", round(-scores["test_score"].mean(), 3))

model.fit(X_train, y_train)
print("test error, once:", round(mean_absolute_error(y_test, model.predict(X_test)), 3))
```

```
cross-validated error: 0.753
test error, once: 0.82
```

The split sets 240 stations aside. Cross-validation judges the model on the other 960 alone, and the
test stations are used once, at the end, for the number to report. Both are mean absolute errors in
degrees Celsius.

## Setup

Three tools from the standard library, the data libraries, and the parts of scikit-learn the
workflow uses.

- `json`, `Path`, `joblib` and `shutil` save the model and its versions at the end, and remove the
  scratch folder afterward
- `numpy` and `pandas` build the stations and hold the report
- `ColumnTransformer`, `SimpleImputer`, `StandardScaler`, `OneHotEncoder`, `Pipeline` and
  `make_pipeline` prepare the columns inside the model
- `DummyRegressor`, `LinearRegression` and `RandomForestRegressor` are the three candidates
- `train_test_split`, `KFold`, `cross_validate` and `GridSearchCV` split, compare and search, and
  `mean_absolute_error` and `r2_score` score the chosen model on the test rows

`station_network` makes the network of 1,200 weather stations in Norway. The stations are made up,
and a generator started from a fixed seed draws them, so the table is the same on every run.
`raw_export` turns the table into what a station register might send: 68 private stations with no
elevation, 28 stations with no owner, and 88 terrains typed in capitals with a trailing space. `raw`
is that export, and `SCRATCH` is a folder for the saved model.

In [1]:
import json
import shutil
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]


def raw_export(network, seed=7):
    """The network as a station register exports it: some private stations' elevations missing,
    terrain typed by hand in several ways, and a few owners left blank."""
    rng = np.random.default_rng(seed)
    raw = network.copy()
    raw["elevation_m"] = raw["elevation_m"].astype(float)
    hidden = (raw["owner"] == "private").to_numpy() & (rng.uniform(size=len(raw)) < 0.3)
    raw.loc[hidden, "elevation_m"] = np.nan
    retyped = rng.uniform(size=len(raw)) < 0.08
    raw.loc[retyped, "terrain"] = raw.loc[retyped, "terrain"].str.upper() + " "
    blank = rng.uniform(size=len(raw)) < 0.02
    raw.loc[blank, "owner"] = np.nan
    return raw


raw = raw_export(network)
SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)

print("scikit-learn", sklearn.__version__, "|", len(raw), "rows in the raw export")

scikit-learn 1.6.1 | 1200 rows in the raw export


## Worked examples

### The raw export

Before choosing anything, count what is missing and look at how each text column was typed:

In [2]:
print(raw.shape)
print(raw.isna().sum()[lambda counts: counts > 0])
print(raw["terrain"].value_counts().sort_index())

(1200, 11)
elevation_m    68
owner          28
dtype: int64
terrain
COASTAL       14
FJORD         31
INLAND        40
MOUNTAIN       3
coastal      128
fjord        399
inland       542
mountain      43
Name: count, dtype: int64


Two columns have gaps, 68 elevations and 28 owners, and `terrain` holds eight spellings of four
values. The counts are sorted by label so that their order does not depend on how the sort breaks
ties.

### Cleaning that learns nothing

Removing the spaces and the capitals treats every row the same way, whatever the other rows hold, so
it can happen before the split. It goes in a function, because every table the model sees later,
including a planned station typed by hand, needs the same treatment:

In [3]:
def clean_text(table):
    """The table with its terrain trimmed and in lowercase, as the model was trained on it."""
    return table.assign(terrain=table["terrain"].str.strip().str.lower())


clean = clean_text(raw)
print(clean["terrain"].value_counts().sort_index())

terrain
coastal     142
fjord       430
inland      582
mountain     46
Name: count, dtype: int64


Four spellings remain. The gaps stay, because filling them means choosing a value from the data, and
that belongs inside the pipeline.

### The target, the features, and the columns left out

The question is a planned station's January mean, so `jan_mean` is the target. The features are what
the office knows before building, the five numbers and the two text columns. `station` is an id, and
`snow_on_may_1` and `june_frost` are measured at a station after it is built, so all three stay out:

In [4]:
NUMERIC = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]
TEXT = ["terrain", "owner"]

X = clean[NUMERIC + TEXT]
y = clean["jan_mean"]
print(X.shape, y.shape)

(1200, 7) (1200,)


### A single split held out until the end

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(len(X_train), "stations to learn and choose with,", len(X_test), "set aside")
print("missing elevations:", X_train["elevation_m"].isna().sum(), "to learn with and",
      X_test["elevation_m"].isna().sum(), "set aside")

960 stations to learn and choose with, 240 set aside
missing elevations: 53 to learn with and 15 set aside


From here on, nothing looks at `X_test` or `y_test` until the model is chosen. The test rows hold 15
of the missing elevations, so the report can say how the model does on stations without one.

### The preparation inside the pipeline

`prepare` builds a new `ColumnTransformer` each time it is called, so every model gets its own,
fitted only on the rows that model is given:

In [6]:
def prepare():
    """Fill and scale the numbers, fill and encode the text."""
    numbers = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    text = make_pipeline(SimpleImputer(strategy="most_frequent"),
                         OneHotEncoder(handle_unknown="ignore"))
    return ColumnTransformer([("numbers", numbers, NUMERIC), ("text", text, TEXT)])


print(prepare().fit(X_train).get_feature_names_out())

['numbers__latitude' 'numbers__longitude' 'numbers__elevation_m'
 'numbers__coast_km' 'numbers__installed' 'text__terrain_coastal'
 'text__terrain_fjord' 'text__terrain_inland' 'text__terrain_mountain'
 'text__owner_airport' 'text__owner_met' 'text__owner_private'
 'text__owner_road']


A missing elevation becomes the median of the training rows, a missing owner the most common one,
and each text value its own column of 0s and 1s. `handle_unknown="ignore"` turns a value the encoder
never saw into a row of zeros instead of an error, which Common errors shows has a cost.

### Comparing candidates by cross-validation

`DummyRegressor` predicts the training rows' mean for every station, which is the simple guess the
office would compare against. Each candidate is cross-validated on the training rows with the same
five folds, and scored by mean absolute error, in degrees:

In [7]:
folds = KFold(5, shuffle=True, random_state=0)
candidates = {
    "mean only": DummyRegressor(),
    "linear": LinearRegression(),
    "forest": RandomForestRegressor(random_state=0),
}

rows, fold_errors = [], {}
for name, estimator in candidates.items():
    model = Pipeline([("prepare", prepare()), ("model", estimator)])
    scores = cross_validate(model, X_train, y_train, cv=folds, scoring="neg_mean_absolute_error")
    errors = -scores["test_score"]
    fold_errors[name] = errors
    rows.append({"model": name, "mean error": errors.mean(), "spread across folds": errors.std()})

lower = int((fold_errors["linear"] < fold_errors["forest"]).sum())
print("folds where the linear model's error is lower than the forest's:", lower, "of 5")
pd.DataFrame(rows).set_index("model").round(3)

folds where the linear model's error is lower than the forest's: 4 of 5


,mean error,spread across folds
model,,
mean only,2.350,0.116
linear,0.753,0.031
forest,0.799,0.021


Both real models beat the mean by a wide margin, 0.753 and 0.799 against 2.35. The linear model's
error is lower on average and in four of the five folds, so cross-validation favors it, by a margin
small enough that the forest deserves a search before it is set aside.

### A small search for the forest

A forest has settings that a default run leaves untried, so before choosing the linear model, a
small grid tries six settings for the forest, still on the training rows only:

In [8]:
search = GridSearchCV(
    Pipeline([("prepare", prepare()), ("model", RandomForestRegressor(random_state=0))]),
    {"model__min_samples_leaf": [1, 5, 20], "model__max_features": [1.0, 0.5]},
    cv=folds, scoring="neg_mean_absolute_error",
)
search.fit(X_train, y_train)

results = pd.DataFrame(search.cv_results_)
print(results[["param_model__min_samples_leaf", "param_model__max_features", "mean_test_score"]]
      .assign(mean_error=lambda table: -table["mean_test_score"]).drop(columns="mean_test_score")
      .sort_values("mean_error").round(3).to_string(index=False))
print("best:", search.best_params_)

 param_model__min_samples_leaf  param_model__max_features  mean_error
                             1                        0.5       0.793
                             1                        1.0       0.799
                             5                        1.0       0.809
                             5                        0.5       0.810
                            20                        1.0       0.917
                            20                        0.5       0.944
best: {'model__max_features': 0.5, 'model__min_samples_leaf': 1}


The best forest's cross-validated error is 0.793, still behind the linear model's 0.753, so the
linear model is the choice. It is also the easier one to explain in the meeting, since each feature
has one coefficient.

### Scoring on the test set once

The chosen model is fitted on all 960 training stations and scored on the 240 set aside, beside the
mean-only guess fitted the same way:

In [9]:
chosen = Pipeline([("prepare", prepare()), ("model", LinearRegression())]).fit(X_train, y_train)
guess = DummyRegressor().fit(X_train, y_train)

predicted = chosen.predict(X_test)
print("mean absolute error:", round(mean_absolute_error(y_test, predicted), 3))
print("R² on the test stations:", round(r2_score(y_test, predicted), 3))
print("mean-only guess:", round(mean_absolute_error(y_test, guess.predict(X_test)), 3))

mean absolute error: 0.82
R² on the test stations: 0.871
mean-only guess: 2.39


The test error, 0.82 degrees, is a little higher than the cross-validated 0.753, and the two need
not agree, since 240 stations are one sample and another split would give another number. What makes
0.82 the number for the meeting is that nothing was chosen with these rows. Trying other models on
them now and keeping the best would turn the test rows into a second round of choosing, and the
number would no longer estimate the error on new stations.

### What the model gets wrong

The average hides where the errors are. Grouped by terrain, with the error as predicted minus
measured:

In [10]:
report = X_test.assign(measured=y_test, predicted=predicted.round(1), error=predicted - y_test)

by_terrain = report.groupby("terrain")["error"].agg(
    stations="count", average_error="mean", mean_absolute_error=lambda errors: errors.abs().mean()
)
by_terrain.round(2)

,stations,average_error,mean_absolute_error
terrain,,,
coastal,21,-0.14,0.92
fjord,89,0.08,0.66
inland,119,-0.09,0.89
mountain,11,-0.46,1.16


Mountain stations are off by 1.16 degrees on average, against 0.66 for stations by a fjord, and only
11 mountain stations were in the test rows, so that number rests on few stations. Stations whose
elevation had to be filled in are the other weak spot:

In [11]:
filled = report["elevation_m"].isna().map({True: "elevation filled in",
                                             False: "elevation on record"})
by_elevation = report.groupby(filled)["error"].agg(
    stations="count", mean_absolute_error=lambda errors: errors.abs().mean()
)
by_elevation.round(2)

,stations,mean_absolute_error
elevation_m,,
elevation filled in,15,1.47
elevation on record,225,0.78


A filled-in elevation is the median of the training rows, and temperature falls with height, so a
station far from the median is predicted badly. Its error is 1.47 degrees, nearly twice the 0.78 of
stations with an elevation on record. The three largest misses on the test rows:

In [12]:
misses = report.assign(miss=report["error"].abs()).nlargest(3, "miss")
misses[["latitude", "elevation_m", "coast_km", "terrain", "measured", "predicted"]]

,latitude,elevation_m,coast_km,terrain,measured,predicted
51,69.70,832.0,136.5,mountain,-13.5,-16.8
867,63.83,492.0,76.9,inland,-5.1,-8.4
533,58.77,323.0,4.8,inland,1.7,-1.6


`nlargest` keeps the first of any tied rows in the table's order, so the three are the same on every
run. All three are inland or in the mountains, and each is predicted more than 3 degrees too cold.

### The report for the meeting

The numbers above, in the sentences someone will repeat:

In [13]:
test_error = mean_absolute_error(y_test, predicted)
guess_error = mean_absolute_error(y_test, guess.predict(X_test))
mountain = by_terrain.loc["mountain"]
unknown = by_elevation.loc["elevation filled in"]

print(f"On {len(X_test)} stations it never saw, the model's January mean is off by "
      f"{test_error:.2f} degrees on average.")
print(f"Guessing the training stations' mean for every station is off by {guess_error:.2f}.")
print(f"It is weakest in the mountains, off by {mountain['mean_absolute_error']:.2f} "
      f"on {mountain['stations']:.0f} stations,")
print(f"and where the elevation is unknown, off by {unknown['mean_absolute_error']:.2f} "
      f"on {unknown['stations']:.0f}.")

On 240 stations it never saw, the model's January mean is off by 0.82 degrees on average.
Guessing the training stations' mean for every station is off by 2.39.
It is weakest in the mountains, off by 1.16 on 11 stations,
and where the elevation is unknown, off by 1.47 on 15.


Each sentence prints a value measured on the test rows, so nothing in the report was typed by hand.

### Saving the model with its versions

Following the **Saving a Model** notebook, the pipeline goes into one file and what it needs into
another, and a planned station is predicted from the loaded copy:

In [14]:
joblib.dump(chosen, SCRATCH / "january.joblib")
about = {
    "model": "january.joblib",
    "target": "jan_mean",
    "features": NUMERIC + TEXT,
    "test_mean_absolute_error": round(test_error, 3),
    "versions": {
        "scikit-learn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__,
        "joblib": joblib.__version__,
    },
}
(SCRATCH / "january.json").write_text(json.dumps(about, indent=2))

loaded = joblib.load(SCRATCH / "january.joblib")
planned = clean_text(pd.DataFrame([{
    "latitude": 69.5, "longitude": 23.0, "elevation_m": 450, "coast_km": 21.0,
    "installed": 2026, "terrain": "Inland", "owner": "met",
}]))
print("planned station:", loaded.predict(planned).round(1))

planned station: [-8.8]


The planned station's terrain was typed `Inland`, and `clean_text` made it `inland` before the model
saw it. The model file does not contain `clean_text`, so the program that loads it has to clean its
input the same way.

## Your turn

Four tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/14-a-complete-workflow-solutions.ipynb).

**1.** Add `RandomForestRegressor(min_samples_leaf=5, random_state=0)` to the candidates as `"forest, leaf 5"`, and print the cross-validated mean error of all four on the training rows.

In [15]:
# your code here

**2.** Print the chosen model's mean absolute error on the test rows for each `owner`, with the number of test stations for each.

In [16]:
# your code here

**3.** Predict the January mean of a planned mountain station at latitude 62.0, longitude 9.5, elevation 1,100 m, 140 km from the coast, installed in 2026, owned by `road`, with its terrain typed `MOUNTAIN `, using `chosen` and `clean_text`.

In [17]:
# your code here

**4.** Fit the chosen pipeline without `elevation_m` among the numbers, and print its cross-validated mean error on the training rows beside the full model's, to see what the elevation is worth.

In [18]:
# your code here

## Common errors

### No error, and a perfect score from the target left among the features

Here the numeric columns are chosen by dropping from the table the columns that are not wanted, as
happens when nobody names the ones to keep, and `jan_mean` was not among those dropped:

In [19]:
not_numbers = ["station", "terrain", "owner", "snow_on_may_1", "june_frost"]
with_target = clean.columns.drop(not_numbers).tolist()
print(with_target)
leaky = ColumnTransformer([
    ("numbers", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), with_target),
    ("text", make_pipeline(SimpleImputer(strategy="most_frequent"),
                           OneHotEncoder(handle_unknown="ignore")), TEXT),
])
X_leaky_train, _, _, _ = train_test_split(clean[with_target + TEXT], y,
                                          test_size=0.2, random_state=42)

model = Pipeline([("prepare", leaky), ("model", LinearRegression())])
scores = cross_validate(model, X_leaky_train, y_train, cv=folds, scoring="neg_mean_absolute_error")
print("cross-validated error:", round(-scores["test_score"].mean(), 3))

['latitude', 'longitude', 'elevation_m', 'coast_km', 'installed', 'jan_mean']
cross-validated error: 0.0


An error of 0.0 degrees means the model copied the answer from its input. No real model of the
weather is perfect, so a perfect score is a reason to look for a column that should not be there.
Name the features you keep, as `NUMERIC` and `TEXT` do, and the target can enter only if you write
its name in the list.

### No error, and a preparation step that has seen the test rows

Here the preparation is fitted on all 1,200 stations before the split's training rows go to the
model:

In [20]:
fitted_on_all = prepare().fit(X)
early = LinearRegression().fit(fitted_on_all.transform(X_train), y_train)

imputer = fitted_on_all.named_transformers_["numbers"].named_steps["simpleimputer"]
print("median elevation filled in:", imputer.statistics_[NUMERIC.index("elevation_m")],
      "| from the training rows alone:", X_train["elevation_m"].median())
early_predictions = early.predict(fitted_on_all.transform(X_test))
print("test error:", round(mean_absolute_error(y_test, early_predictions), 3))

median elevation filled in: 252.5 | from the training rows alone: 247.0
test error: 0.819


The median that fills a missing elevation came from all 1,200 stations, the 240 test stations among
them, so the test rows helped shape what the model was given. The test error barely moved, 0.819
against the pipeline's 0.82, because adding 240 rows to 960 moved that median only from 247 to 252.5
meters. The **Pipelines** notebook shows a case, choosing columns from noise, where the same mistake
decides the result. Keep every step that learns inside the pipeline, so that cross-validation and
the test rows score the model on rows that none of its steps learned from.

### ValueError: columns are missing: {'owner'}

In [21]:
no_owner = clean_text(pd.DataFrame([{
    "latitude": 69.5, "longitude": 23.0, "elevation_m": 450, "coast_km": 21.0,
    "installed": 2026, "terrain": "inland",
}]))

chosen.predict(no_owner)

ValueError: columns are missing: {'owner'}

The planned station was typed without its owner, and the pipeline's `ColumnTransformer` expects
every column it was fitted with. It raises at prediction time, so no answer goes out for a station
whose owner was never sent. If the owner is unknown, send the column with a missing value, `np.nan`,
and the imputer fills it as it filled the 28 in the export, as the next cell shows:

In [22]:
print(chosen.predict(no_owner.assign(owner=np.nan)).round(1))

[-8.8]


### No error, and a planned station's terrain ignored

The same mountain station, once with its terrain cleaned and once as it was typed:

In [23]:
typed = pd.DataFrame([{
    "latitude": 69.5, "longitude": 23.0, "elevation_m": 900, "coast_km": 21.0,
    "installed": 2026, "terrain": "MOUNTAIN ", "owner": "met",
}])

print("as typed:", chosen.predict(typed).round(1))
print("cleaned: ", chosen.predict(clean_text(typed)).round(1))
print("terrain columns as typed:", chosen.named_steps["prepare"].transform(typed)[0, 5:9])

as typed: [-11.1]
cleaned:  [-12.3]
terrain columns as typed: [0. 0. 0. 0.]


The encoder had never seen `MOUNTAIN ` with a capital and a space, and `handle_unknown="ignore"`
turned it into four zeros, so the model treated the station as having no terrain at all and
predicted 1.2 degrees warmer than for a mountain. Nothing warned. Pass every new table through
`clean_text` before the model, as the saved model's own example did.

The notebook is finished with its files, so this cell removes the scratch folder:

In [24]:
shutil.rmtree(SCRATCH)

print("scratch still there:", SCRATCH.exists())

scratch still there: False


## Recap

- Clean what needs no learning before the split, in a function every later table goes through
- Name the features to keep, split once, and keep every step that learns inside a pipeline
- Choose between models by cross-validation on the training rows, and score the choice on the test
  rows once
- Report the error beside a simple guess, and where the model does worst
- Save the pipeline with its versions, and clean a new station's input the same way

## What is next

The **PyTorch and TensorFlow, Deep Dive** guide builds models that learn their own features, and
trains them with the same habits of splitting, scoring and saving.

---

&#8592; **Previous:** [Saving a Model](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/13-saving-a-model.ipynb)  &nbsp;·&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)
